# Spatial Statistics & Regression in Python

This notebook demonstrates:
- Testing for spatial autocorrelation (Moran's I)
- Creating LISA cluster maps
- Running OLS regression and checking residuals
- Spatial lag and spatial error regression models
- Interpreting and visualizing results

**Use this for your capstone project** if you're doing regression analysis.

---

## Before you start

Read the [Spatial Statistics reading](../readings/spatial-statistics.md) for conceptual background.

---

## Step 0: Setup

In [ ]:
# Detect environment and install packages
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Installing spatial statistics packages (takes 2-3 minutes)...")
    !pip install geopandas libpysal esda spreg splot mapclassify -q
    print("Done!")
else:
    print("Running locally. Make sure you have: conda activate intro-gis")
    print("If packages are missing: conda install -c conda-forge libpysal esda spreg splot")

In [ ]:
# Import libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Spatial statistics
from libpysal.weights import Queen, KNN
from esda.moran import Moran, Moran_Local
from spreg import OLS, ML_Lag, ML_Error
from splot.esda import moran_scatterplot, lisa_cluster

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

---

## Step 1: Set up folder structure and load data

### Folder Structure

For this capstone notebook, we'll use:
- **RAW** = `intro-gis/capstone/data/raw/` - for reading input data
- **PROCESSED** = `intro-gis/capstone/data/processed/` - for saving outputs (maps, results, processed datasets)

**For your capstone:** 
- Place your original data files in `capstone/data/raw/`
- All outputs (cleaned data, maps, tables) should be saved to `capstone/data/processed/`

In [ ]:
# Set up paths based on environment
if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Capstone paths in Google Drive
    RAW = Path('/content/drive/MyDrive/intro-gis/capstone/data/raw')
    PROCESSED = Path('/content/drive/MyDrive/intro-gis/capstone/data/processed')
else:
    # Local paths (relative to notebook location)
    RAW = Path('data/raw')
    PROCESSED = Path('data/processed')

# Create directories if they don't exist
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Data folders ready:")
print(f"  Input data: {RAW.absolute()}")
print(f"  Output data: {PROCESSED.absolute()}")

# Load sample data - US counties with socioeconomic variables
# This is a built-in dataset from PySAL
from libpysal import examples

# Load the NAT dataset (US counties, natural amenities)
nat_path = examples.get_path('NAT.shp')
gdf = gpd.read_file(nat_path)

print(f"\nLoaded {len(gdf)} counties")
print(f"\nColumns available:")
print(gdf.columns.tolist())

# FOR YOUR CAPSTONE: Replace the above with your own data like this:
# gdf = gpd.read_file(RAW / 'your_data.shp')
# or
# gdf = pd.read_csv(RAW / 'your_data.csv')

In [ ]:
# Select variables for our analysis
# HR90 = Homicide rate 1990
# UE90 = Unemployment rate 1990  
# DV90 = Divorce rate 1990
# MA90 = Median age 1990

# Rename for clarity
gdf = gdf.rename(columns={
    'HR90': 'homicide_rate',
    'UE90': 'unemployment',
    'DV90': 'divorce_rate',
    'MA90': 'median_age'
})

# Check for missing values
print("Missing values:")
print(gdf[['homicide_rate', 'unemployment', 'divorce_rate', 'median_age']].isnull().sum())

# Drop rows with missing values for this example
gdf = gdf.dropna(subset=['homicide_rate', 'unemployment', 'divorce_rate', 'median_age'])
print(f"\n{len(gdf)} counties after removing missing values")

In [ ]:
# Map the outcome variable
fig, ax = plt.subplots(figsize=(12, 8))

gdf.plot(
    column='homicide_rate',
    scheme='quantiles',
    k=5,
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'title': 'Homicide Rate'},
    ax=ax
)

ax.set_title('Homicide Rate by County (1990)', fontsize=14)
ax.set_axis_off()

# Save the map
output_path = PROCESSED / 'homicide_rate_map.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Map saved to: {output_path}")

plt.show()

print("\nNotice: High values appear to cluster in the South. Let's test if this is significant.")

---

## Step 2: Create spatial weights

Spatial weights define which locations are "neighbours." We need this for all spatial statistics.

In [ ]:
# Create Queen contiguity weights
# Queen = neighbours share an edge OR corner
w = Queen.from_dataframe(gdf)

# Row-standardize (each row sums to 1)
# This is standard practice for Moran's I
w.transform = 'r'

print(f"Spatial weights created!")
print(f"Average number of neighbours: {w.mean_neighbors:.1f}")
print(f"Min neighbours: {w.min_neighbors}")
print(f"Max neighbours: {w.max_neighbors}")

In [ ]:
# Check for "islands" (locations with no neighbours)
# These can cause problems in spatial regression

if w.islands:
    print(f"Warning: {len(w.islands)} islands (no neighbours) found!")
    print("Consider using KNN weights instead, or removing these locations.")
else:
    print("No islands found. All locations have at least one neighbour.")

---

## Step 3: Test for spatial autocorrelation (Moran's I)

Before running regression, test if your outcome variable is spatially autocorrelated.

In [ ]:
# Global Moran's I for homicide rate
moran = Moran(gdf['homicide_rate'], w)

print("=" * 50)
print("GLOBAL MORAN'S I TEST")
print("=" * 50)
print(f"Moran's I:    {moran.I:.4f}")
print(f"Expected I:   {moran.EI:.4f}  (if random)")
print(f"p-value:      {moran.p_sim:.4f}  (permutation test)")
print("=" * 50)

if moran.p_sim < 0.05:
    if moran.I > 0:
        print("\nInterpretation: Significant POSITIVE spatial autocorrelation.")
        print("High values cluster near high values, low near low.")
    else:
        print("\nInterpretation: Significant NEGATIVE spatial autocorrelation.")
        print("High values tend to be near low values.")
else:
    print("\nInterpretation: No significant spatial autocorrelation.")
    print("The pattern could be random.")

In [ ]:
# Moran scatterplot
# X-axis: standardized values
# Y-axis: spatial lag (average of neighbours' standardized values)

fig, ax = moran_scatterplot(moran, aspect_equal=True)
ax.set_title(f"Moran Scatterplot (I = {moran.I:.3f}, p = {moran.p_sim:.4f})")

# Save the plot
output_path = PROCESSED / 'moran_scatterplot.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Moran scatterplot saved to: {output_path}")

plt.show()

print("\nThe positive slope confirms positive spatial autocorrelation.")
print("Points in upper-right (HH) and lower-left (LL) quadrants drive this pattern.")

---

## Step 4: LISA cluster map

LISA (Local Indicators of Spatial Association) shows WHERE clustering occurs.

In [ ]:
# Calculate Local Moran's I
lisa = Moran_Local(gdf['homicide_rate'], w)

# Add cluster classifications to geodataframe
# q values: 1=HH, 2=LH, 3=LL, 4=HL
gdf['lisa_cluster'] = lisa.q
gdf['lisa_pvalue'] = lisa.p_sim

# Create significance mask (p < 0.05)
gdf['significant'] = gdf['lisa_pvalue'] < 0.05

# Count clusters
cluster_labels = {1: 'High-High', 2: 'Low-High', 3: 'Low-Low', 4: 'High-Low'}
sig_clusters = gdf[gdf['significant']]['lisa_cluster'].value_counts()

print("Significant LISA clusters:")
for q, count in sig_clusters.items():
    print(f"  {cluster_labels[q]}: {count} counties")

In [ ]:
# Create LISA cluster map
fig, ax = plt.subplots(figsize=(12, 8))

# Define colors for each cluster type
colors = {
    0: 'lightgrey',   # Not significant
    1: 'red',         # High-High
    2: 'lightblue',   # Low-High 
    3: 'blue',        # Low-Low
    4: 'pink'         # High-Low
}

# Create color column (0 for not significant)
gdf['cluster_color'] = gdf.apply(
    lambda row: row['lisa_cluster'] if row['significant'] else 0, 
    axis=1
)

# Plot
gdf.plot(
    column='cluster_color',
    categorical=True,
    cmap='coolwarm',
    legend=True,
    ax=ax
)

ax.set_title('LISA Cluster Map: Homicide Rate', fontsize=14)
ax.set_axis_off()

# Manual legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='High-High (Hot Spot)'),
    Patch(facecolor='blue', label='Low-Low (Cold Spot)'),
    Patch(facecolor='pink', label='High-Low (Outlier)'),
    Patch(facecolor='lightblue', label='Low-High (Outlier)'),
    Patch(facecolor='lightgrey', label='Not Significant')
]
ax.legend(handles=legend_elements, loc='lower left')

# Save the map
output_path = PROCESSED / 'lisa_cluster_map.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"LISA cluster map saved to: {output_path}")

plt.show()

print("\nRed (Hot Spots): Counties with high homicide rates surrounded by high rates")
print("Blue (Cold Spots): Counties with low rates surrounded by low rates")

---

## Step 5: OLS regression (baseline)

First, run standard OLS regression to establish a baseline. Then check if residuals are spatially autocorrelated.

In [ ]:
# Prepare data for regression
# y = outcome variable (column vector)
# X = predictor variables (matrix)

y = gdf[['homicide_rate']].values
X = gdf[['unemployment', 'divorce_rate', 'median_age']].values

# Run OLS regression with spatial diagnostics
ols = OLS(
    y, X, w=w,
    name_y='homicide_rate',
    name_x=['unemployment', 'divorce_rate', 'median_age'],
    spat_diag=True  # Include spatial diagnostics
)

print(ols.summary)

In [ ]:
# Extract and display key statistics in a cleaner format
print("=" * 60)
print("OLS REGRESSION RESULTS")
print("=" * 60)
print(f"R-squared: {ols.r2:.4f}")
print(f"Adjusted R-squared: {ols.ar2:.4f}")
print(f"Log-Likelihood: {ols.logll:.2f}")
print(f"AIC: {ols.aic:.2f}")
print("=" * 60)
print("\nCoefficients:")
print("-" * 60)
print(f"{'Variable':<20} {'Coef':>10} {'Std.Err':>10} {'t-stat':>10} {'p-value':>10}")
print("-" * 60)
for i, name in enumerate(['CONSTANT'] + ['unemployment', 'divorce_rate', 'median_age']):
    print(f"{name:<20} {ols.betas[i][0]:>10.4f} {ols.std_err[i]:>10.4f} {ols.t_stat[i][0]:>10.4f} {ols.t_stat[i][0]:>10.4f}")

In [ ]:
# Check OLS residuals for spatial autocorrelation
# This is the CRITICAL step - if residuals are autocorrelated, OLS is problematic

residuals = ols.u.flatten()
moran_resid = Moran(residuals, w)

print("=" * 50)
print("SPATIAL AUTOCORRELATION IN OLS RESIDUALS")
print("=" * 50)
print(f"Moran's I of residuals: {moran_resid.I:.4f}")
print(f"p-value: {moran_resid.p_sim:.4f}")
print("=" * 50)

if moran_resid.p_sim < 0.05:
    print("\n⚠️  SIGNIFICANT spatial autocorrelation in residuals!")
    print("    OLS results may be biased. Spatial regression is needed.")
else:
    print("\n✓ No significant spatial autocorrelation in residuals.")
    print("  OLS results are probably fine.")

In [ ]:
# Map the residuals to see spatial pattern
gdf['ols_residuals'] = residuals

fig, ax = plt.subplots(figsize=(12, 8))

gdf.plot(
    column='ols_residuals',
    scheme='quantiles',
    k=5,
    cmap='RdBu_r',
    legend=True,
    ax=ax
)

ax.set_title('OLS Residuals (Red = underpredicted, Blue = overpredicted)', fontsize=14)
ax.set_axis_off()

# Save the map
output_path = PROCESSED / 'ols_residuals_map.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Residuals map saved to: {output_path}")

plt.show()

print("\nIf you see spatial clustering of red or blue, residuals are spatially autocorrelated.")

---

## Step 6: Spatial regression models

Since OLS residuals are spatially autocorrelated, we need spatial regression.

In [ ]:
# SPATIAL LAG MODEL
# Assumes: outcome depends on neighbours' outcomes (spatial spillover)
# Example: homicide rate in one county affects neighbouring counties

lag_model = ML_Lag(
    y, X, w,
    name_y='homicide_rate',
    name_x=['unemployment', 'divorce_rate', 'median_age']
)

print("=" * 60)
print("SPATIAL LAG MODEL RESULTS")
print("=" * 60)
print(lag_model.summary)

In [ ]:
# SPATIAL ERROR MODEL
# Assumes: errors are spatially correlated (unmeasured spatially-clustered variables)
# Example: missing variables (neighbourhood quality, etc.) are spatially clustered

error_model = ML_Error(
    y, X, w,
    name_y='homicide_rate',
    name_x=['unemployment', 'divorce_rate', 'median_age']
)

print("=" * 60)
print("SPATIAL ERROR MODEL RESULTS")
print("=" * 60)
print(error_model.summary)

In [ ]:
# COMPARE MODELS
# Lower AIC = better fit (penalizing for complexity)

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(f"{'Model':<25} {'Log-Likelihood':>15} {'AIC':>15}")
print("-" * 60)
print(f"{'OLS':<25} {ols.logll:>15.2f} {ols.aic:>15.2f}")
print(f"{'Spatial Lag':<25} {lag_model.logll:>15.2f} {lag_model.aic:>15.2f}")
print(f"{'Spatial Error':<25} {error_model.logll:>15.2f} {error_model.aic:>15.2f}")
print("=" * 60)

# Find best model
models = {'OLS': ols.aic, 'Spatial Lag': lag_model.aic, 'Spatial Error': error_model.aic}
best = min(models, key=models.get)
print(f"\nBest model (lowest AIC): {best}")

# Save model comparison to CSV
comparison_df = pd.DataFrame({
    'Model': ['OLS', 'Spatial Lag', 'Spatial Error'],
    'Log-Likelihood': [ols.logll, lag_model.logll, error_model.logll],
    'AIC': [ols.aic, lag_model.aic, error_model.aic]
})
output_path = PROCESSED / 'model_comparison.csv'
comparison_df.to_csv(output_path, index=False)
print(f"\nModel comparison saved to: {output_path}")

In [ ]:
# Check if spatial model fixed the autocorrelation
# Use the best model's residuals

lag_residuals = lag_model.u.flatten()
moran_lag_resid = Moran(lag_residuals, w)

print("=" * 50)
print("RESIDUAL CHECK: SPATIAL LAG MODEL")
print("=" * 50)
print(f"Moran's I of residuals: {moran_lag_resid.I:.4f}")
print(f"p-value: {moran_lag_resid.p_sim:.4f}")
print("=" * 50)

if moran_lag_resid.p_sim >= 0.05:
    print("\n✓ No significant spatial autocorrelation in spatial lag residuals.")
    print("  The spatial model successfully accounted for spatial dependence.")
else:
    print("\n⚠️  Residuals still show spatial autocorrelation.")
    print("  Consider alternative model specifications.")

---

## Step 7: Interpret and report results

In [ ]:
# Create a comparison table for your report

print("=" * 80)
print("REGRESSION RESULTS COMPARISON TABLE")
print("=" * 80)
print(f"\n{'Variable':<20} {'OLS':>15} {'Spatial Lag':>15} {'Spatial Error':>15}")
print("-" * 80)

vars = ['CONSTANT', 'unemployment', 'divorce_rate', 'median_age']
for i, var in enumerate(vars):
    ols_coef = f"{ols.betas[i][0]:.3f}"
    lag_coef = f"{lag_model.betas[i][0]:.3f}"
    err_coef = f"{error_model.betas[i][0]:.3f}"
    print(f"{var:<20} {ols_coef:>15} {lag_coef:>15} {err_coef:>15}")

# Spatial parameters
print(f"{'rho (spatial lag)':<20} {'-':>15} {lag_model.rho:.3f}***{' ':>8} {'-':>15}")
print(f"{'lambda (error)':<20} {'-':>15} {'-':>15} {error_model.lam:.3f}***")

print("-" * 80)
print(f"{'R-squared':<20} {ols.r2:>15.3f} {lag_model.pr2:>15.3f} {error_model.pr2:>15.3f}")
print(f"{'AIC':<20} {ols.aic:>15.1f} {lag_model.aic:>15.1f} {error_model.aic:>15.1f}")
print("=" * 80)
print("\n*** p < 0.001")

# Save results table to CSV
results_df = pd.DataFrame({
    'Variable': vars + ['rho (spatial lag)', 'lambda (error)', 'R-squared', 'AIC'],
    'OLS': [f"{ols.betas[i][0]:.3f}" for i in range(4)] + ['-', '-', f"{ols.r2:.3f}", f"{ols.aic:.1f}"],
    'Spatial Lag': [f"{lag_model.betas[i][0]:.3f}" for i in range(4)] + [f"{lag_model.rho:.3f}***", '-', f"{lag_model.pr2:.3f}", f"{lag_model.aic:.1f}"],
    'Spatial Error': [f"{error_model.betas[i][0]:.3f}" for i in range(4)] + ['-', f"{error_model.lam:.3f}***", f"{error_model.pr2:.3f}", f"{error_model.aic:.1f}"]
})
output_path = PROCESSED / 'regression_results.csv'
results_df.to_csv(output_path, index=False)
print(f"\nRegression results table saved to: {output_path}")

In [ ]:
# Example interpretation for your capstone report

print("""
EXAMPLE INTERPRETATION FOR YOUR REPORT:
========================================

We examined the relationship between county-level socioeconomic factors and 
homicide rates in the United States using spatial regression analysis.

SPATIAL AUTOCORRELATION:
Initial testing revealed significant positive spatial autocorrelation in 
homicide rates (Moran's I = {moran_i:.3f}, p < 0.001), indicating that counties 
with high homicide rates tend to cluster near other high-rate counties. LISA 
analysis identified hot spots primarily in the South and cold spots in the 
Midwest and Northeast.

MODEL SELECTION:
OLS regression residuals showed significant spatial autocorrelation 
(Moran's I = {moran_resid_i:.3f}, p < 0.05), violating independence assumptions. 
We therefore estimated spatial lag and spatial error models. Based on AIC 
comparison, the spatial lag model provided the best fit (AIC = {lag_aic:.1f} 
vs. {ols_aic:.1f} for OLS).

KEY FINDINGS:
The spatial lag coefficient (rho = {rho:.3f}, p < 0.001) indicates significant 
spatial spillover effects—homicide rates in one county are positively associated 
with rates in neighbouring counties, even after controlling for socioeconomic 
factors.

After accounting for spatial dependence:
- Unemployment remains positively associated with homicide rates (b = X.XX)
- Divorce rate shows [describe relationship]
- Median age shows [describe relationship]

LIMITATIONS:
This analysis is cross-sectional and cannot establish causation. The significant 
spatial lag term could reflect true spillover effects or omitted variables that 
are spatially clustered.
""".format(
    moran_i=moran.I,
    moran_resid_i=moran_resid.I,
    lag_aic=lag_model.aic,
    ols_aic=ols.aic,
    rho=lag_model.rho
))

---

## Summary: What to do for your capstone

1. **Set up your folder structure**
   - Place original data in `capstone/data/raw/`
   - All outputs will be saved to `capstone/data/processed/`

2. **Map your outcome variable** - Look for visible clustering
   - Save maps to PROCESSED folder for your report

3. **Test for spatial autocorrelation** (Moran's I)
   - If not significant, you might be fine with standard methods
   - If significant, consider why and proceed with spatial methods

4. **Create LISA maps** to show where clusters are
   - Save cluster maps to PROCESSED folder

5. **Run OLS first** as a baseline

6. **Check OLS residuals** for spatial autocorrelation
   - If not significant, OLS is probably fine
   - If significant, you need spatial regression
   - Save residual maps to PROCESSED folder

7. **Compare spatial models** using AIC
   - Save comparison tables to PROCESSED folder

8. **Check spatial model residuals** to confirm the model worked

9. **Interpret substantively** - What do the results mean for your research question?

10. **Save all outputs**
    - Maps: PROCESSED / 'your_map_name.png'
    - Tables: PROCESSED / 'your_table_name.csv'
    - Processed data: PROCESSED / 'your_data_name.shp' or .csv

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`

**All outputs are saved to:** `capstone/data/processed/` for easy access in your capstone report!